### Load Dataset


In [17]:
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()

### Import Libraries


In [18]:
import numpy as np
import pandas as pd

from pytorch_tabular import TabularModel
from pytorch_tabular.models import CategoryEmbeddingModelConfig
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig

In [19]:
# Convert the data to a pandas DataFrame
data = pd.DataFrame(cancer.data, columns=cancer.feature_names)
data['diagnosis'] = cancer.target
data

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


In [20]:
data.shape

(569, 31)

In [21]:
def load_data(df, target_col, test_size):
    torch_data = np.array(df.drop(target_col, axis=1))
    torch_labels = np.array(df[target_col])
    data = np.hstack([torch_data, torch_labels.reshape(-1, 1)])
    gen_names = [f"feature_{i}" for i in range(data.shape[-1])]
    col_names = gen_names
    col_names[-1] = "target"
    data = pd.DataFrame(data, columns=col_names)
    cat_col_names = [x for x in gen_names[:-1] if len(data[x].unique()) < 10]
    num_col_names = [x for x in gen_names[:-1] if x not in [target_col] + cat_col_names]
    test_idx = data.sample(int(test_size * len(data)), random_state=42).index
    test = data[data.index.isin(test_idx)]
    train = data[~data.index.isin(test_idx)]
    
    return (train, test, ["target"], cat_col_names, num_col_names)

In [ ]:
train, test, target_col, cat_col_names, num_col_names= load_data(data, 'diagnosis', 0.2)

### Setting Up Configs


In [25]:
data_config = DataConfig(
    target=['target'],
    continuous_cols=num_col_names,
    categorical_cols=cat_col_names,
    continuous_feature_transform="quantile_normal",
    normalize_continuous_features=True
)

trainer_config = TrainerConfig(
    auto_lr_find=True,
    batch_size=32,
    max_epochs=100,
    # early_stopping_patience=3
)

optimizer_config = OptimizerConfig()
model_config = CategoryEmbeddingModelConfig(
    task="classification",
    layers="4096-4096-512",
    activation="LeakyReLU",
    learning_rate = 1e-3,
    metrics=["accuracy"]
)

tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
)

2025-01-08 01:17:39,932 - {pytorch_tabular.tabular_model:146} - INFO - Experiment Tracking is turned off

In [27]:
tabular_model.fit(train=train)

Seed set to 42


2025-01-08 01:18:30,631 - {pytorch_tabular.tabular_model:548} - INFO - Preparing the DataLoaders

2025-01-08 01:18:30,638 - {pytorch_tabular.tabular_datamodule:522} - INFO - Setting up the datamodule for          
classification task

c:\Users\amits\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2785: UserWarning: n_quantiles (1000) is greater than the total number of samples (365). n_quantiles is set to n_samples.
  warnings.warn(


2025-01-08 01:18:30,714 - {pytorch_tabular.tabular_model:599} - INFO - Preparing the Model: CategoryEmbeddingModel

...

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │ 19.0 M │ train │
│ 1 │ _embedding_layer │ Embedding1dLayer          │     60 │ train │
│ 2 │ head             │ LinearHead                │  1.0 K │ train │
│ 3 │ loss             │ CrossEntropyLoss          │      0 │ train │
└───┴──────────────────┴───────────────────────────┴────────┴───────┘

Trainable params: 19.0 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 19.0 M                                                                                               
Total estimated model params size (MB): 76                                                                         
Modules in train mode: 16                                                                                          
Modules in eval mode: 0

Output()

: 

: 

In [ ]:
tabular_model.evaluate(test)